# Bria AI Training Notebook (ONLINE ONLY)

This notebook is pre-configured for the **bria-ai-training** repository with a focus on **Online Only services**. It will install Unsloth, load your conversation data, train the model, and export it for use on your phone or PC.

In [ ]:
# 1. Install Unsloth and dependencies
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# 2. Load the Model
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
# 3. Load Your Data (already in the repo)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template = "llama-3.1")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# Path to the file in your repo
dataset = load_dataset("json", data_files="Calls/training_output/training_chat_finetune.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print("Data loaded from repo!")

In [ ]:
# 4. Set up Trainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 120,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer.train()

In [ ]:
# 5. Save to GGUF
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")